In [ ]:
import requests
import json
header = {'User-Agent':'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36'}

response = requests.get("https://image.baidu.com/search/acjson?tn=resultjson_com&logid=3345384492595205401&ipn=rj&"
                        "ct=201326592&is=&fp=result&fr=ala&word=%E7%8C%AB%E5%92%AA%E5%9B%BE%E7%89%87&queryWord="
                        "%E7%8C%AB%E5%92%AA%E5%9B%BE%E7%89%87&cl=2&lm=-1&ie=utf-8&oe=utf-8&adpicid=&st=&z=&ic=&hd=&latest=&copyright="
                        "&s=&se=&tab=&width=&height=&face=&istype=&qc=&nc=&expermode=&nojc=&isAsync=&pn=100&rn=100&gsm=3c&1662186069094=",
                        headers=header)

# response = requests.get("https://image.baidu.com/search/acjson?tn=resultjson_com&word=%E5%B0%8F%E7%8B%97&ie=utf-8&fp=result&fr=&ala=0&applid=9216922044399147381&pn=30&rn=30&nojc=0&gsm=1e&newReq=1"
#                         # "ct=201326592&is=&fp=result&fr=ala&word=%E7%8C%AB%E5%92%AA%E5%9B%BE%E7%89%87&queryWord="
#                         "ct=201326592&is=&fp=result&fr=ala&word=%E5%B0%8F%E7%8B%97%E5%9B%BE%E7%89%87&queryWord="
#                         "%E5%B0%8F%E7%8B%97%E5%9B%BE%E7%89%87&cl=2&lm=-1&ie=utf-8&oe=utf-8&adpicid=&st=&z=&ic=&hd=&latest=&copyright="
#                         "&s=&se=&tab=&width=&height=&face=&istype=&qc=&nc=&expermode=&nojc=&isAsync=&pn=10&rn=200&gsm=3c&1662186069094=",
#                         headers=header)


data = response.text
obj = json.loads(data)
starList = obj['data']

x = 1
for item in starList:
    if 'thumbURL' in item:
        print(item['thumbURL'])
        responseIMG = requests.get(item['thumbURL'])
        content = responseIMG.content   # 获取图片内容,二进制流
        with open('./img/' + str(x) + '.jpg', 'wb') as f:
            f.write(content)
        x = x + 1

In [8]:
import requests
import json
import time
import random
import os
from urllib.parse import quote

def baidu_image_spider(keyword, download_num, save_dir='./img'):
    """
    百度图片爬虫函数
    :param keyword: 搜索关键词，如 '猫咪'
    :param download_num: 想要下载的图片数量
    :param save_dir: 图片保存目录
    """
    
    # 创建保存目录
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # 请求头
    header = {
        'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36'
    }
    
    # 用于去重的集合
    downloaded_urls = set()
    downloaded_count = 400
    page_size = 60  # 百度每页返回的大概数量
    pn = 0  # 起始页码
    
    while downloaded_count < download_num:
        # 编码关键词
        encoded_keyword = quote(keyword)
        
        # 修复URL格式 - 使用能成功获取数据的格式
        url = f"https://image.baidu.com/search/acjson?tn=resultjson_com&logid=3345384492595205401&ipn=rj&" \
              f"ct=201326592&is=&fp=result&fr=ala&word={encoded_keyword}&queryWord=" \
              f"{encoded_keyword}&cl=2&lm=-1&ie=utf-8&oe=utf-8&adpicid=&st=&z=&ic=&hd=&latest=&copyright=" \
              f"&s=&se=&tab=&width=&height=&face=&istype=&qc=&nc=&expermode=&nojc=&isAsync=&pn={pn}&rn={page_size}&gsm=3c&{int(time.time()*1000)}="
        
        try:
            print(f"正在请求第 {pn//page_size + 1} 页...")
            response = requests.get(url, headers=header, timeout=10)
            response.encoding = 'utf-8'
            
            # 处理JSON数据
            data = response.text
            
            try:
                obj = json.loads(data)
                image_list = obj.get('data', [])
                
                if not image_list:
                    print("没有更多图片了")
                    break
                
                print(f"本页获取到 {len(image_list)} 张图片")
                
                for item in image_list:
                    if downloaded_count >= download_num:
                        break
                    
                    # 获取图片URL（优先使用thumbURL）
                    thumbURL = item.get('thumbURL') or item.get('middleURL') or item.get('objURL')
                    
                    if thumbURL and thumbURL not in downloaded_urls:
                        downloaded_urls.add(thumbURL)
                        
                        try:
                            # 下载图片
                            print(f"正在下载第 {downloaded_count + 1} 张图片...")
                            responseIMG = requests.get(thumbURL, headers=header, timeout=15)
                            
                            if responseIMG.status_code == 200:
                                # 生成唯一文件名
                                file_ext = '.jpg'  # 默认扩展名
                                if 'webp' in thumbURL.lower():
                                    file_ext = '.webp'
                                elif 'png' in thumbURL.lower():
                                    file_ext = '.png'
                                elif 'gif' in thumbURL.lower():
                                    file_ext = '.gif'
                                
                                filename = f"{downloaded_count + 1}{file_ext}"
                                filepath = os.path.join(save_dir, filename)
                                
                                with open(filepath, 'wb') as f:
                                    f.write(responseIMG.content)
                                
                                downloaded_count += 1
                                
                                
                                # 随机延迟，避免请求过快
                                # time.sleep(random.uniform(0.5, 1.5))
                                
                            else:
                                print(f"下载失败，状态码: {responseIMG.status_code}")
                                
                        except Exception as e:
                            print(f"下载图片时出错: {e}")
                            continue
                
            except json.JSONDecodeError as e:
                print(f"JSON解析错误: {e}")
                print(f"原始数据: {data[:200]}...")
                break
                
        except Exception as e:
            print(f"请求错误: {e}")
            break
        
        # 翻到下一页
        pn += page_size
        
        # 页面间延迟
        time.sleep(random.uniform(1, 2))
    
    print(f"\n下载完成！共下载 {downloaded_count} 张图片到目录: {save_dir}")

# 使用示例
if __name__ == "__main__":
    # 在这里设置你的参数
    search_keyword = "派大星章鱼哥"  # 搜索关键词 - 修改为中文而非编码
    desired_count = 500     # 想要下载的图片数量 - 先测试少量
    save_directory = "./image_set/sponge_bob_images"  # 保存目录

    # 开始爬取
    baidu_image_spider(search_keyword, desired_count, save_directory)

正在请求第 1 页...
JSON解析错误: Invalid \escape: line 229 column 165 (char 184823)
原始数据: {"queryEnc":"%C5%C9%B4%F3%D0%C7%D5%C2%D3%E3%B8%E7","queryExt":"派大星章鱼哥","queryFeature":"CgoKBsXJtPPQxxACCgoKBtXC0-O45xAC",
"aiEditDataSample": {    "IMAGE_PC_B_AIGC_HOVER": "",    "IMAGE_PC_C_AIGC_HOVE...

下载完成！共下载 400 张图片到目录: ./image_set/sponge_bob_images


In [1]:
import requests
import json
import time
import random
import os
import re


def baidu_image_spider(keyword, download_num, save_dir='./img'):
    """
    百度图片爬虫函数
    :param keyword: 搜索关键词，如 '猫咪'
    :param download_num: 想要下载的图片数量
    :param save_dir: 图片保存目录
    """

    # 创建保存目录
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 请求头
    header = {
        'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36',
        'Accept': 'application/json, text/plain, */*'
    }

    # 用于去重的集合
    downloaded_urls = set()
    downloaded_count = 1100
    page_size = 60  # 百度每页返回的大概数量
    pn = 0  # 起始页码

    def extract_urls_by_regex(text: str):
        # 兼容 JSON 中的转义斜杠 \/
        candidates = []
        for key in ["thumbURL", "middleURL", "objURL"]:
            candidates += re.findall(rf'"{key}":"(http[^"\\]+)"', text)
            candidates += [u.replace('\\/', '/') for u in re.findall(rf'"{key}":"(http[^"\\]+)"', text)]
        # 去重且保持顺序
        seen = set()
        ordered = []
        for u in candidates:
            if u not in seen:
                seen.add(u)
                ordered.append(u)
        return ordered

    while downloaded_count < download_num:
        # 使用 params 构造查询，自动编码，避免长关键词导致的 URL 组装错误
        params = {
            'tn': 'resultjson_com',
            'ipn': 'rj',
            'ct': '201326592',
            'fp': 'result',
            'queryWord': keyword,
            'word': keyword,
            'cl': '2',
            'lm': '-1',
            'ie': 'utf-8',
            'oe': 'utf-8',
            'pn': pn,
            'rn': page_size,
            'gsm': '3c'
        }

        try:
            print(f"正在请求第 {pn // page_size + 1} 页...")
            response = requests.get(
                'https://image.baidu.com/search/acjson',
                headers=header,
                params=params,
                timeout=10
            )
            response.encoding = 'utf-8'
            data = response.text

            image_items = None
            try:
                # 优先尝试标准 JSON 解析
                obj = response.json()
                image_items = obj.get('data', []) if isinstance(obj, dict) else []
            except Exception as e:
                print(f"JSON解析错误: {e}")
                # 降级：正则从文本中抽取 URL，避免因非法转义导致的失败
                urls = extract_urls_by_regex(data)
                if not urls:
                    print("无法从响应中提取图片URL，结束。")
                    break
                # 转为统一的数据结构，便于后续处理
                image_items = [{"thumbURL": u} for u in urls]

            if not image_items:
                print("没有更多图片了")
                break

            print(f"本页获取到 {len(image_items)} 条记录")

            for item in image_items:
                if downloaded_count >= download_num:
                    break

                # 获取图片URL（优先使用thumbURL）
                thumbURL = item.get('thumbURL') or item.get('middleURL') or item.get('objURL')

                if not thumbURL:
                    continue

                # 规整可能的转义
                thumbURL = thumbURL.replace('\\/', '/')

                if thumbURL in downloaded_urls:
                    continue
                downloaded_urls.add(thumbURL)

                try:
                    # 下载图片
                    print(f"正在下载第 {downloaded_count + 1} 张图片...")
                    responseIMG = requests.get(thumbURL, headers=header, timeout=15)

                    if responseIMG.status_code == 200:
                        # 生成唯一文件名
                        lower_url = thumbURL.lower()
                        if '.webp' in lower_url or 'webp' in lower_url:
                            file_ext = '.webp'
                        elif '.png' in lower_url or 'png' in lower_url:
                            file_ext = '.png'
                        elif '.gif' in lower_url or 'gif' in lower_url:
                            file_ext = '.gif'
                        else:
                            file_ext = '.jpg'

                        filename = f"{downloaded_count + 1}{file_ext}"
                        filepath = os.path.join(save_dir, filename)

                        with open(filepath, 'wb') as f:
                            f.write(responseIMG.content)

                        downloaded_count += 1

                        # 随机延迟，避免请求过快（如需要再开启）
                        # time.sleep(random.uniform(0.5, 1.5))

                    else:
                        print(f"下载失败，状态码: {responseIMG.status_code}")

                except Exception as e:
                    print(f"下载图片时出错: {e}")
                    continue

        except Exception as e:
            print(f"请求错误: {e}")
            break

        # 翻到下一页
        pn += page_size

        # 页面间延迟
        time.sleep(random.uniform(1, 2))

    print(f"\n下载完成！共下载 {downloaded_count} 张图片到目录: {save_dir}")


# 使用示例
if __name__ == "__main__":
    # 在这里设置你的参数
    search_keyword = "痞老板"  # 搜索关键词
    desired_count = 1200     # 想要下载的图片数量
    save_directory = "./image_set/sponge_bob_images"  # 保存目录

    # 开始爬取
    baidu_image_spider(search_keyword, desired_count, save_directory)

正在请求第 1 页...
本页获取到 61 条记录
正在下载第 1101 张图片...
正在下载第 1102 张图片...
正在下载第 1103 张图片...
正在下载第 1104 张图片...
下载图片时出错: HTTPSConnectionPool(host='img0.baidu.com', port=443): Read timed out. (read timeout=15)
正在下载第 1104 张图片...
正在下载第 1105 张图片...
正在下载第 1106 张图片...
正在下载第 1107 张图片...
正在下载第 1108 张图片...
正在下载第 1109 张图片...
正在下载第 1110 张图片...
正在下载第 1111 张图片...
正在下载第 1112 张图片...
正在下载第 1113 张图片...
正在下载第 1114 张图片...
正在下载第 1115 张图片...
正在下载第 1116 张图片...
正在下载第 1117 张图片...
正在下载第 1118 张图片...
正在下载第 1119 张图片...
正在下载第 1120 张图片...
正在下载第 1121 张图片...
正在下载第 1122 张图片...
正在下载第 1123 张图片...
正在下载第 1124 张图片...
正在下载第 1125 张图片...
正在下载第 1126 张图片...
正在下载第 1127 张图片...
正在下载第 1128 张图片...
正在下载第 1129 张图片...
正在下载第 1130 张图片...
正在下载第 1131 张图片...
正在下载第 1132 张图片...
正在下载第 1133 张图片...
正在下载第 1134 张图片...
正在下载第 1135 张图片...
正在下载第 1136 张图片...
正在下载第 1137 张图片...
正在下载第 1138 张图片...
正在下载第 1139 张图片...
正在下载第 1140 张图片...
下载图片时出错: ('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None))
正在下载第 1140 张图片...
下载图片时出错: HTTPSConne